In [1]:
import kagglehub
path = kagglehub.dataset_download("quadeer15sh/augmented-forest-segmentation")
print("Path to dataset files:", path)

d:\Sirius26\cv_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\popyg\.cache\kagglehub\datasets\quadeer15sh\augmented-forest-segmentation\versions\2


In [2]:
!pip install segmentation-models-pytorch albumentations tqdm

error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [7]:
import os, re, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import segmentation_models_pytorch as smp
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import json
import os
from tqdm import tqdm


# from google.colab import drive
# drive.mount('/content/drive')

ModuleNotFoundError: No module named 'albumentations'

In [ ]:
path = "Forest Segmented/Forest Segmented/"
src_images = os.path.join(path, 'images')
src_masks = os.path.join(path, 'masks')

if not os.path.exists(src_images):
    for root, dirs, files in os.walk(path):
        if 'images' in dirs:
            src_images = os.path.join(root, 'images')
        if 'masks' in dirs:
            src_masks = os.path.join(root, 'masks')
    print(f"Найдены images: {src_images}")
    print(f"Найдены masks: {src_masks}")

dst_root = "/home/prihoslo/Documents/vs/ds-phase-2/09-cv/chekpoint/forest_seg_split"
for split in ['train', 'valid']:
    os.makedirs(os.path.join(dst_root, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dst_root, split, 'masks'), exist_ok=True)

def extract_id(filename):
    match = re.search(r'(\d+)', filename)
    return match.group(1) if match else None

mask_dict = {}
for mask_file in os.listdir(src_masks):
    mid = extract_id(mask_file)
    if mid:
        mask_dict[mid] = mask_file

pairs = []
for img_file in os.listdir(src_images):
    img_id = extract_id(img_file)
    if img_id and img_id in mask_dict:
        pairs.append((img_file, mask_dict[img_id]))
    else:
        print(f"Пропускаем {img_file}: нет маски")

print(f"Найдено пар: {len(pairs)}")
if len(pairs) == 0:
    raise RuntimeError("Не найдено ни одной пары.")

train_pairs, valid_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
print(f"Train пар: {len(train_pairs)}, Valid пар: {len(valid_pairs)}")

def copy_pairs(pair_list, split_name):
    for img_file, mask_file in pair_list:
        shutil.copy2(os.path.join(src_images, img_file),
                     os.path.join(dst_root, split_name, 'images', img_file))
        shutil.copy2(os.path.join(src_masks, mask_file),
                     os.path.join(dst_root, split_name, 'masks', img_file))
    print(f"Скопировано {len(pair_list)} пар в {split_name}")

copy_pairs(train_pairs, 'train')
copy_pairs(valid_pairs, 'valid')
print(f"Данные готовы в {dst_root}")

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, root, transform=None):
        self.images_dir = os.path.join(root, 'images')
        self.masks_dir = os.path.join(root, 'masks')
        self.image_names = sorted(os.listdir(self.images_dir))
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, img_name)

        image = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('L'))

        if mask.max() > 1:
            mask = (mask / 255).astype(np.int64)

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        else:
            image = torch.from_numpy(image).permute(2,0,1).float() / 255.0
            mask = torch.from_numpy(mask).long()

        return image, mask

In [ ]:
IMAGE_SIZE = 256
batch_size = 64

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=15, p=0.5),
    A.RandomScale(scale_limit=0.1, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.5),
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    ToTensorV2(),
])

valid_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    ToTensorV2(),
])

train_dataset = SegmentationDataset(os.path.join(dst_root, 'train'), transform=train_transform)
valid_dataset = SegmentationDataset(os.path.join(dst_root, 'valid'), transform=valid_transform)


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=6)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=6)

print(f"Train samples: {len(train_dataset)}, Valid samples: {len(valid_dataset)}")

In [ ]:
# drive_folder = "/content/drive/MyDrive/forest_seg_checkpoints"
# local_folder = "/content/forest_seg_checkpoints"
# if os.path.exists(drive_folder):
#     shutil.rmtree(local_folder, ignore_errors=True)
#     shutil.copytree(drive_folder, local_folder)
#     print("✅ Папка с чекпоинтами скопирована из Google Диска!")
# else:
#     raise FileNotFoundError("Папка forest_seg_checkpoints не найдена на Google Диске. Убедитесь, что ярлык добавлен.")

# # -------------------- 4. Модель, оптимизатор, loss --------------------
# num_classes = 1
# model = smp.Unet(
#     encoder_name="tu-convnext_base",
#     encoder_weights=None,
#     in_channels=3,
#     classes=num_classes,
#     activation=None
# )
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# loss_fn = smp.losses.DiceLoss(mode='binary') + nn.BCEWithLogitsLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)

# def compute_metrics(pred_bin, target):
#     pred_bin = pred_bin.long()
#     target = target.long()
#     tp = ((pred_bin == 1) & (target == 1)).sum().float()
#     tn = ((pred_bin == 0) & (target == 0)).sum().float()
#     fp = ((pred_bin == 1) & (target == 0)).sum().float()
#     fn = ((pred_bin == 0) & (target == 1)).sum().float()
#     smooth = 1e-6
#     iou = (tp + smooth) / (tp + fp + fn + smooth)
#     dice = (2*tp + smooth) / (2*tp + fp + fn + smooth)
#     precision = (tp + smooth) / (tp + fp + smooth)
#     recall = (tp + smooth) / (tp + fn + smooth)
#     accuracy = (tp + tn + smooth) / (tp + tn + fp + fn + smooth)
#     return iou, dice, precision, recall, accuracy

# # -------------------- 5. Загрузка последнего или лучшего чекпоинта и истории --------------------
# best_checkpoint_path = os.path.join(local_folder, "best_unet_full.pth")
# last_checkpoint_path = os.path.join(local_folder, "last_unet_full.pth")
# history_path = os.path.join(local_folder, "segmentation_history.json")

# # Определяем, какой чекпоинт загружать (последний приоритетнее)
# if os.path.exists(last_checkpoint_path):
#     checkpoint_path = last_checkpoint_path
#     print("✅ Найден последний чекпоинт (last_unet_full.pth)")
# elif os.path.exists(best_checkpoint_path):
#     checkpoint_path = best_checkpoint_path
#     print("✅ Найден лучший чекпоинт (best_unet_full.pth)")
# else:
#     checkpoint_path = None
#     print("🆕 Чекпоинты не найдены, начинаем обучение с нуля")

# if checkpoint_path is not None:
#     checkpoint = torch.load(checkpoint_path, map_location=device)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#     start_epoch = checkpoint['epoch']          # эпоха, на которой сохранён чекпоинт (уже завершена)
#     best_valid_iou = checkpoint.get('valid_iou', 0.0)
#     print(f"✅ Чекпоинт загружен. Возобновляем обучение с эпохи {start_epoch}. Лучший IoU: {best_valid_iou:.4f}")
# else:
#     start_epoch = 0
#     best_valid_iou = 0.0

# # Загружаем историю (если есть)
# if os.path.exists(history_path):
#     with open(history_path, 'r') as f:
#         history = json.load(f)
#     print(f"✅ История загружена: {len(history['epochs'])} эпох")
# else:
#     history = {
#         'epochs': [], 'train_loss': [], 'val_loss': [], 'val_iou': [],
#         'val_dice': [], 'val_precision': [], 'val_recall': [], 'val_accuracy': []
#     }
#     print("🆕 История не найдена, создана новая")

# # -------------------- 6. Цикл обучения с сохранением лучшего и последнего чекпоинтов --------------------
# num_epochs = 30

# for epoch in range(start_epoch, num_epochs):
#     model.train()
#     train_loss = 0.0
#     loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
#     for images, masks in loop:
#         images, masks = images.to(device), masks.to(device)
#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = loss_fn(outputs, masks.unsqueeze(1).float())
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item() * images.size(0)
#         loop.set_postfix(loss=loss.item())
#     train_loss /= len(train_loader.dataset)

#     model.eval()
#     valid_loss = 0.0
#     sum_iou = sum_dice = sum_precision = sum_recall = sum_accuracy = 0.0
#     with torch.no_grad():
#         for images, masks in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]"):
#             images, masks = images.to(device), masks.to(device)
#             outputs = model(images)
#             loss = loss_fn(outputs, masks.unsqueeze(1).float())
#             valid_loss += loss.item() * images.size(0)

#             pred_mask = (torch.sigmoid(outputs) > 0.5).squeeze(1)
#             iou, dice, precision, recall, accuracy = compute_metrics(pred_mask, masks)
#             b = images.size(0)
#             sum_iou += iou * b
#             sum_dice += dice * b
#             sum_precision += precision * b
#             sum_recall += recall * b
#             sum_accuracy += accuracy * b

#     total = len(valid_loader.dataset)
#     valid_loss /= total
#     valid_iou = sum_iou / total
#     valid_dice = sum_dice / total
#     valid_precision = sum_precision / total
#     valid_recall = sum_recall / total
#     valid_accuracy = sum_accuracy / total

#     scheduler.step(valid_loss)

#     print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Valid Loss={valid_loss:.4f}, IoU={valid_iou:.4f}, Dice={valid_dice:.4f}, Prec={valid_precision:.4f}, Rec={valid_recall:.4f}, Acc={valid_accuracy:.4f}")

#     # ----- Сохранение истории (после каждой эпохи) -----
#     history['epochs'].append(epoch+1)
#     history['train_loss'].append(train_loss)
#     history['val_loss'].append(valid_loss)
#     history['val_iou'].append(float(valid_iou))
#     history['val_dice'].append(float(valid_dice))
#     history['val_precision'].append(float(valid_precision))
#     history['val_recall'].append(float(valid_recall))
#     history['val_accuracy'].append(float(valid_accuracy))

#     with open(history_path, 'w') as f:
#         json.dump(history, f)

#     # ----- Сохраняем последний чекпоинт (после каждой эпохи) -----
#     torch.save({
#         'epoch': epoch+1,
#         'model_state_dict': model.state_dict(),
#         'optimizer_state_dict': optimizer.state_dict(),
#         'scheduler_state_dict': scheduler.state_dict(),
#         'valid_iou': valid_iou,
#         'valid_loss': valid_loss,
#     }, last_checkpoint_path)

#     # ----- Сохраняем лучшую модель (при улучшении IoU) -----
#     if valid_iou > best_valid_iou:
#         best_valid_iou = valid_iou
#         torch.save({
#             'epoch': epoch+1,
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'scheduler_state_dict': scheduler.state_dict(),
#             'valid_iou': valid_iou,
#             'valid_loss': valid_loss,
#         }, best_checkpoint_path)
#         print(f"  -> Лучшая модель обновлена (IoU={valid_iou:.4f})")

# print(f"✅ Обучение завершено. История сохранена в {history_path}")

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Комбинированный loss (Dice + BCE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.1)

# ------------------------------------------------------------
# Функция для расчёта метрик (должна быть определена ранее)
# ------------------------------------------------------------
def compute_metrics(pred_bin, target):
    pred_bin = pred_bin.long()
    target = target.long()
    tp = ((pred_bin == 1) & (target == 1)).sum().float()
    tn = ((pred_bin == 0) & (target == 0)).sum().float()
    fp = ((pred_bin == 1) & (target == 0)).sum().float()
    fn = ((pred_bin == 0) & (target == 1)).sum().float()
    smooth = 1e-6
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    dice = (2*tp + smooth) / (2*tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    accuracy = (tp + tn + smooth) / (tp + tn + fp + fn + smooth)
    return iou, dice, precision, recall, accuracy

# ------------------------------------------------------------
# Параметры
# ------------------------------------------------------------
checkpoint_dir = "/home/prihoslo/Documents/vs/ds-phase-2/09-cv/chekpoint"
os.makedirs(checkpoint_dir, exist_ok=True)
history_path = os.path.join(checkpoint_dir, "segmentation_history.json")

if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)
    print(f"✅ Загружена существующая история ({len(history['epochs'])} эпох)")
else:
    history = {
        'epochs': [], 'train_loss': [], 'val_loss': [], 'val_iou': [],
        'val_dice': [], 'val_precision': [], 'val_recall': [], 'val_accuracy': []
    }
    print("🆕 Создана новая история обучения")

# Чекпоинты
best_checkpoint_path = os.path.join(checkpoint_dir, "best_exp.pth")
last_checkpoint_path = os.path.join(checkpoint_dir, "last_exp.pth")
num_epochs = 20
best_valid_iou = 0.0

# Цикл обучения
for epoch in range(num_epochs):
    # Обучение
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, masks in loop:
        images, masks = images.to(device), masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks.unsqueeze(1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    train_loss /= len(train_loader)

    # Валидация
    model.eval()
    valid_loss = 0.0
    sum_iou = sum_dice = sum_precision = sum_recall = sum_accuracy = 0.0
    with torch.no_grad():
        for images, masks in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            masks = masks.unsqueeze(1).float()
            loss = criterion(outputs, masks)
            valid_loss += loss.item()
            pred_mask = (torch.sigmoid(outputs) > 0.5).squeeze(1)
            iou, dice, precision, recall, accuracy = compute_metrics(pred_mask, masks)
            b = images.size(0)
            sum_iou += iou.item() * b
            sum_dice += dice.item() * b
            sum_precision += precision.item() * b
            sum_recall += recall.item() * b
            sum_accuracy += accuracy.item() * b

    total = len(valid_loader.dataset)
    valid_loss /= len(valid_loader)
    valid_iou = sum_iou / total
    valid_dice = sum_dice / total
    valid_precision = sum_precision / total
    valid_recall = sum_recall / total
    valid_accuracy = sum_accuracy / total

    scheduler.step(valid_loss)   # планировщик по валидационной потере

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Valid Loss={valid_loss:.4f}, IoU={valid_iou:.4f}, Dice={valid_dice:.4f}, Prec={valid_precision:.4f}, Rec={valid_recall:.4f}, Acc={valid_accuracy:.4f}")

    # Сохранение истории
    history['epochs'].append(epoch+1)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(valid_loss)
    history['val_iou'].append(float(valid_iou))
    history['val_dice'].append(float(valid_dice))
    history['val_precision'].append(float(valid_precision))
    history['val_recall'].append(float(valid_recall))
    history['val_accuracy'].append(float(valid_accuracy))
    with open(history_path, 'w') as f:
        json.dump(history, f)

    # Сохранение последнего чекпоинта
    torch.save({
        'epoch': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'valid_iou': valid_iou,
        'valid_loss': valid_loss,
    }, last_checkpoint_path)

    # Сохранение лучшей модели
    if valid_iou > best_valid_iou:
        best_valid_iou = valid_iou
        torch.save({
            'epoch': epoch+1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'valid_iou': valid_iou,
            'valid_loss': valid_loss,
        }, best_checkpoint_path)
        print(f"  -> Лучшая модель обновлена (IoU={valid_iou:.4f})")

print(f"✅ Обучение завершено. История сохранена в {history_path}")

NameError: name 'smp' is not defined

In [ ]:
# import torch
# torch.cuda.empty_cache()